# Analyze Policy Packages Across Runs

This notebook analyzes scenario batches generated from full policy combinations and produces comparable figures/tables across runs.

**Main inputs**
- `runs/complete_interactions/<run_id>/*/output.csv`
- `runs/complete_interactions/<run_id>/policies/indicator.csv`

**Main outputs**
- figures in `runs/complete_interactions/<run_id>/img/`
- merged indicator table: `img/indicator.csv`


## Usage

1. Set `folder` and `assessment` in the configuration cell.
2. Run all cells from top to bottom.
3. Retrieve generated figures from the `img` folder.

`assessment` options:
- `market_failures`
- `policies`
- `optimal_pp`


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

import sys
sys.path.insert(0, "../../../..")

from project.analysis.post_processing.policy_decomposition.helpers import (
    load_scenario_outputs,
    make_interaction_heatmap,
    make_interaction_heatmap_percent,
    make_mapping_scatter,
    make_swarmplot,
    resolve_run_folder,
)
from project.utils import (
    get_json,
    horizontal_stack_bar_plot,
    manual_shapley_analysis,
    manual_sobol_analysis,
    horizontal_waterfall_chart,
    vertical_waterfall_chart,
)
from project.write_output import indicator_policies, export_policy_latex_table

warnings.filterwarnings("ignore")

## Configuration


In [ ]:
# Select the run folder and analysis mode.
# folder, assessment = "policies_scenarios_market_failures", "market_failures"
# folder, assessment = "policies_scenarios_optimal_pp", "optimal_pp"
folder, assessment = "policies_scenarios_policies", "policies"
# folder, assessment = "policies_scenarios_20260326_200257", "optimal_pp"

scenarios_description_file = "policies_scenarios_description.csv"
policies_to_drop = [] # ["subsidy_status_quo", "subsidy_present_bias"]

folder = str(resolve_run_folder(folder, scenarios_description_file))
folder_out = os.path.join(folder, "img")
os.makedirs(folder_out, exist_ok=True)

print(f"Input folder: {folder}")
print(f"Assessment mode: {assessment}")


## Load Data


In [ ]:
scenarios_description = (
    pd.read_csv(os.path.join(folder, scenarios_description_file), index_col=[0])
    .rename_axis("Scenarios")
)
list_features = scenarios_description.columns.tolist()
scenarios_description = scenarios_description.astype(str)

dict_output = load_scenario_outputs(folder)

print(f"Loaded {len(dict_output)} scenario output files.")


In [ ]:
reference = "Reference"
config_policies = get_json("project/input/policies/cba_inputs.json")

_ = indicator_policies(dict_output, folder, config_policies, reference=reference, figure=False)

indicator_df = pd.read_csv(os.path.join(folder, "policies/indicator.csv"), index_col=[0]).T
data = pd.concat((scenarios_description, indicator_df), axis=1)
data.to_csv(os.path.join(folder_out, "indicator.csv"))


## Group Scenarios

The next cell builds scenario groups used throughout the plotting sections.


In [ ]:
if assessment == "optimal_pp":
    selected_policies_to_drop = [column for column in policies_to_drop if column in data.columns]
    for column in selected_policies_to_drop:
        data = data.loc[data[column] != column, :]
        data.drop(columns=column, inplace=True)
        list_features = [feature for feature in list_features if feature != column]


def _scenarios_with_prefix(feature_names, prefix, negate=False):
    idx = data.index
    for feature in feature_names:
        feature_prefix = data[feature].astype(str).str.split("_").str[0]
        mask = feature_prefix != prefix if negate else feature_prefix == prefix
        idx = idx.intersection(data[mask].index)
    return idx


data["Group"] = "Other"

idx_all_no = _scenarios_with_prefix(list_features, "no")
if assessment == "market_failures":
    data.loc[idx_all_no, "Group"] = "All friction"
elif assessment in ["policies", "optimal_pp"]:
    data.loc[idx_all_no, "Group"] = "No policy"

idx_all_active = _scenarios_with_prefix(list_features, "no", negate=True)
if assessment == "market_failures":
    data.loc[idx_all_active, "Group"] = "No friction"
elif assessment == "optimal_pp":
    data.loc[idx_all_active, "Group"] = "Second best subsidies"
    data = data[data["subsidy_emission"] != "carbon_tax_vsc"]

# Standalone effects: one active feature while all others are "no_*".
for feature in list_features:
    for feature_value in data[feature].dropna().unique():
        if feature_value.split("_")[0] == "no":
            continue
        idx = data.index.intersection(data[data[feature] == feature_value].index)
        for other_feature in [candidate for candidate in list_features if candidate != feature]:
            other_prefix = data[other_feature].astype(str).str.split("_").str[0]
            idx = idx.intersection(data[other_prefix == "no"].index)
        data.loc[idx, "Group"] = feature_value

additional_scenarios = {
    "CarbonTax": "SCC Benchmark",
    "CurrentPolicies": "Package 2018",
    "Package2021": "Package 2021",
    "Package2024": "Baseline Package",
    "Package2024Ban": "Baseline Package + Ban",
    "NoMF": "No friction",
}
for scenario_name, group_name in additional_scenarios.items():
    data.loc[data.index == scenario_name, "Group"] = group_name

data["Size"] = 100
data.loc[data["Group"] != "Other", "Size"] = 200


## Reference Targets

Keep these targets if you want to draw objective lines on the mapping charts.


In [ ]:
objective_carbon_neutrality = 523
emission_reference = dict_output[reference].loc["Emission (MtCO2)"].sum()
emission_saving_objective = emission_reference - objective_carbon_neutrality

# Keep objective lines disabled by default.
energy_saving_objective = None
emission_saving_objective = None


## Format Labels and Colors


In [ ]:
dict_replace = {
    "ghg_externality": "GHG externalities",
    "carbon_tax_vsc": "SCC Benchmark",
    "carbon_tax_high": "Carbon tax",
    "carbon_tax": "Carbon tax",
    "subsidy_emission": "Subsidies emission",
    "subsidy_health_cost": "Subsidies health cost",
    "subsidy_landlord": "Subsidies landlord",
    "subsidy_multi_family": "Subsidies multi family",
    "subsidy_present_bias": "Subsidies present bias",
    "subsidy_status_quo": "Subsidies status quo",
    "tax_status_quo": "Tax status quo",
    "subsidy_credit": "Subsidies credit constraint", 
    "subsidy_hp_statusquo": "Subsidies HP status quo",
    "landlord_dilemma": "Landlord dilemma",
    "multi_family": "Multi family friction",
    "health_cost": "Health cost",
    "credit_constraint": "Credit constraint",
    "credit_constraint_heater": "Credit constraint heater",
    "credit_constraint_insulation": "Credit constraint insulation",
    "present_bias": "Present bias",
    "status_quo_bias": "Status quo bias",
    "subsidies": "Subsidies",
    "subsidies_2018": "Subsidies 2018",
    "subsidies_2021": "Subsidies 2021",
    "subsidies_2024": "Subsidies 2024",
    "restriction_gas": "Ban gas",
    "cee": "WCO",
    "cee_2018": "WCO 2018",
    "cee_2021": "WCO 2021",
    "cee_2024": "WCO 2024",
    "zero_interest_loan": "Zero interest loan",
    "obligation": "Rental ban",
}
data = data.replace(dict_replace)

color_dict = {
    "GHG externalities": "darkgreen",
    "Subsidies emission": "darkgreen",
    "Carbon tax": "darkgreen",
    "Carbon tax high": "darkgreen",
    "Carbon tax + EU-ETS 2": "darkgreen",
    "SCC Benchmark": "darkgreen",
    "Landlord dilemma": "orange",
    "Subsidies landlord": "orange",
    "Multi family friction": "darkmagenta",
    "Subsidies multi family": "darkmagenta",
    "Credit constraint": "red",
    "Credit constraint insulation": "red",
    "Credit constraint heater": "red",
    "Present bias": "blue",
    "Subsidies present bias": "blue",
    "Status quo bias": "purple",
    "Subsidies credit constraint": "red",
    "Subsidies HP status quo": "purple",
    "Subsidies status quo": "purple",
    "Tax status quo": "purple",
    "Subsidies health cost": "yellow",
    "Other": "lightgrey",
    "No policy": "black",
    "All friction": "black",
    "No friction": "green",
    "Second best subsidies": "green",
    "Health cost": "yellow",
    "Subsidies": "royalblue",
    "Subsidies 2018": "royalblue",
    "Subsidies 2021": "royalblue",
    "Subsidies 2024": "royalblue",
    "Ban gas": "red",
    "WCO": "darkorange",
    "WCO 2018": "darkorange",
    "WCO 2021": "darkorange",
    "WCO 2024": "darkorange",
    "White certificate obligation": "orange",
    "Zero interest loan": "darkmagenta",
    "Rental ban": "firebrick",
    "Package 2018": "royalblue",
    "Package 2021": "darkorange",
    "Baseline Package": "firebrick",
    "Baseline Package + Ban": "darkmagenta",
    "P2018": "royalblue",
    "P2021": "darkorange",
    "P2024": "firebrick",
    "P2024 + Ban": "darkmagenta",
    "First best": "green",
    "Second best": "blue",
    "Third best": "red",
    "Carbon tax marginal damage": "black",
    "Combination": "grey",
}


## Mapping Plots

Scatter plots for welfare, energy savings, and emission savings.


In [ ]:
def mapping(**kwargs):
    return make_mapping_scatter(data=data, color_dict=color_dict, **kwargs)


In [ ]:
def run_mapping_suite():
    format_int = lambda value, _: f"{value:.0f}"
    format_two_decimals = lambda value, _: f"{value:.2f}"

    if assessment == "policies":
        annotate = [
            "Package 2018",
            "Package 2021",
            "Baseline Package",
            "Baseline Package + Ban",
            "No policy",
            "SCC Benchmark",
            "No friction",
            "P2018",
            "P2021",
            "P2024",
            "P2024 + Ban",
        ]
        remove_legend = ["Other"]
        x_label = "Social welfare compared to No policy (billion euro per year)"

        mapping(
            x="NPV annual (Billion euro/year)",
            y="Cumulated energy saving (TWh)",
            hue="Group",
            file=os.path.join(folder_out, "mapping_energy_efficiency_gap_policies.pdf"),
            format_y=format_int,
            format_x=format_int,
            annotate=annotate,
            x_label=x_label,
            remove_legend=remove_legend,
            legend_title="Policies standalone",
            size="Size",
            hline=energy_saving_objective,
        )
        mapping(
            x="NPV annual (Billion euro/year)",
            y="Cumulated emission saving (MtCO2)",
            hue="Group",
            file=os.path.join(folder_out, "mapping_emission_efficiency_gap_policies.pdf"),
            format_y=format_int,
            format_x=format_int,
            annotate=annotate,
            x_label=x_label,
            remove_legend=remove_legend,
            legend_title="Policies standalone",
            size="Size",
            hline=emission_saving_objective,
        )
        mapping(
            x="Investment / energy savings (euro/kWh)",
            y="Cumulated energy saving (TWh)",
            hue="Group",
            file=os.path.join(folder_out, "mapping_cost_efficiency_kwh_policies.pdf"),
            format_y=format_int,
            format_x=format_two_decimals,
            annotate=annotate,
            x_label=x_label,
            remove_legend=remove_legend,
            legend_title="Policies standalone",
            size="Size",
            hline=emission_saving_objective,
        )
        mapping(
            x="Investment / emission (euro/tCO2)",
            y="Cumulated emission saving (MtCO2)",
            hue="Group",
            file=os.path.join(folder_out, "mapping_cost_efficiency_tCO2_policies.pdf"),
            format_y=format_int,
            format_x=format_int,
            annotate=annotate,
            x_label="Investment / emission (euro/tCO2)",
            remove_legend=remove_legend,
            legend_title="Policies standalone",
            size="Size",
            hline=emission_saving_objective,
        )

    elif assessment == "market_failures":
        annotate = ["No friction", "All friction"]
        remove_legend = ["Other"]
        x_label = "Social welfare compared to No friction (billion euro per year)"

        mapping(
            x="NPV annual (Billion euro/year)",
            y="Cumulated energy saving (TWh)",
            hue="Group",
            file=os.path.join(folder_out, "mapping_energy_efficiency_gap_mf.pdf"),
            format_y=format_int,
            format_x=format_int,
            annotate=annotate,
            x_label=x_label,
            remove_legend=remove_legend,
            legend_title="Market failures standalone",
            size="Size",
            hline=energy_saving_objective,
        )
        mapping(
            x="NPV annual (Billion euro/year)",
            y="Cumulated emission saving (MtCO2)",
            hue="Group",
            file=os.path.join(folder_out, "mapping_emission_efficiency_gap_mf.pdf"),
            format_y=format_int,
            format_x=format_int,
            annotate=annotate,
            x_label=x_label,
            remove_legend=remove_legend,
            legend_title="Market failures standalone",
            size="Size",
            hline=emission_saving_objective,
        )

    elif assessment == "optimal_pp":
        annotate = ["No policy", "Second best subsidies", "Baseline Package", "No friction"]
        remove_legend = ["Other"]
        x_label = "Cost-benefits analysis (billion euro per year)"

        mapping(
            x="NPV annual (Billion euro/year)",
            y="Cumulated energy saving (TWh)",
            hue="Group",
            file=os.path.join(folder_out, "mapping_energy_efficiency_gap_optimal_pp.pdf"),
            format_y=format_int,
            format_x=format_int,
            annotate=annotate,
            x_label=x_label,
            remove_legend=remove_legend,
            legend_title="Policies standalone",
            size="Size",
            hline=energy_saving_objective,
        )
        mapping(
            x="NPV annual (Billion euro/year)",
            y="Cumulated emission saving (MtCO2)",
            hue="Group",
            file=os.path.join(folder_out, "mapping_emission_efficiency_gap_optimal_pp.pdf"),
            format_y=format_int,
            format_x=format_int,
            annotate=annotate,
            x_label=x_label,
            remove_legend=remove_legend,
            legend_title="Policies standalone",
            size="Size",
            hline=emission_saving_objective,
        )


run_mapping_suite()


## Interaction Analysis

Quantify standalone effects and interactions across policy combinations.


In [ ]:
def interaction(key="NPV annual (Billion euro/year)"):
    results = {}

    for feature in list_features:
        candidate_policies = [item for item in data[feature].unique() if pd.notna(item)]
        for policy in candidate_policies:
            if policy.split("_")[0] == "no":
                continue

            for group_key, group_df in data.groupby([column for column in list_features if column != feature]):
                if not isinstance(group_key, tuple):
                    group_key = (group_key,)

                if key == "NPV annual (Billion euro/year)":
                    value = float(group_df.loc[group_df[feature] == policy, key]) - float(
                        group_df.loc[group_df[feature] == f"no_{feature}", key]
                    )
                elif key in [
                    "Investment / emission (euro/tCO2)",
                    "Investment / energy savings (euro/kWh)",
                ]:
                    numerator_key = "Investment total WT (Billion euro)"
                    numerator = float(group_df.loc[group_df[feature] == policy, numerator_key]) - float(
                        group_df.loc[group_df[feature] == f"no_{feature}", numerator_key]
                    )

                    if key == "Investment / emission (euro/tCO2)":
                        denominator_key = "Emission (MtCO2)"
                    elif key == "Investment / energy savings (euro/kWh)":
                        denominator_key = "Consumption (TWh)"
                    else:
                        raise ValueError("key not recognized")

                    denominator = -(
                        float(group_df.loc[group_df[feature] == policy, denominator_key])
                        - float(group_df.loc[group_df[feature] == f"no_{feature}", denominator_key])
                    )
                    value = numerator / denominator
                    if key == "Investment / emission (euro/tCO2)":
                        value *= 1e3
                else:
                    raise ValueError("key not recognized")

                group_name = group_key
                count_no = len([item for item in group_key if str(item).split("_")[0] == "no"])
                if count_no == len(group_key):
                    group_name = "No policy"
                elif count_no == len(group_key) - 1:
                    group_name = [item for item in group_key if str(item).split("_")[0] != "no"][0]

                results[(policy, group_name)] = value

    rslt = pd.Series(results)
    rslt = rslt.reset_index().set_axis(["policy", "group", "value"], axis=1)
    rslt["hue"] = "Combination"
    rslt.loc[rslt["group"] == "No policy", "hue"] = "No policy"
    return rslt


In [ ]:
orders = None
if assessment == "policies":
    orders = [
        "Carbon tax",
        "Carbon tax + EU-ETS 2",
        "Subsidies 2018",
        "Subsidies 2021",
        "Subsidies 2024",
        "WCO 2018",
        "WCO 2021",
        "WCO 2024",
        "Zero interest loan",
        "Rental ban",
        "Ban gas",
    ]


def run_interaction_suite(
    metric,
    swarm_file,
    swarm_xlabel,
    heatmap_file,
    heatmap_fmt,
    heatmap_xlabel,
    heatmap_percent_file=None,
    swarm_xlim=None,
):
    rslt = interaction(metric)

    xmin = xmax = None
    if swarm_xlim is not None:
        xmin, xmax = swarm_xlim

    make_swarmplot(
        rslt,
        swarm_file,
        xlabel=swarm_xlabel,
        orders=orders,
        xmin=xmin,
        xmax=xmax,
    )

    temp = rslt[rslt["group"].apply(lambda value: isinstance(value, str))].copy()
    temp.loc[temp["group"] == "No policy", "group"] = temp.loc[temp["group"] == "No policy", "policy"]

    make_interaction_heatmap(
        temp,
        heatmap_file,
        fmt=heatmap_fmt,
        xlabel=heatmap_xlabel,
        orders=orders,
    )

    if heatmap_percent_file is not None:
        make_interaction_heatmap_percent(
            temp,
            heatmap_percent_file,
            xlabel=heatmap_xlabel,
            orders=orders,
        )


run_interaction_suite(
    metric="NPV annual (Billion euro/year)",
    swarm_file=os.path.join(folder_out, "assessment_swarmplot_npv.pdf"),
    swarm_xlabel="Social welfare",
    heatmap_file=os.path.join(folder_out, "interactions_heatmap_npv.pdf"),
    heatmap_fmt=".2f",
    heatmap_xlabel="Social welfare (billion euro per year)",
    heatmap_percent_file=os.path.join(folder_out, "interactions_heatmap_percent_npv.pdf"),
)

run_interaction_suite(
    metric="Investment / emission (euro/tCO2)",
    swarm_file=os.path.join(folder_out, "assessment_swarmplot_cost_efficiency_tco2.pdf"),
    swarm_xlabel="Investment / emission avoided (euro/tCO2)",
    heatmap_file=os.path.join(folder_out, "interactions_cost_efficiency_tco2.pdf"),
    heatmap_fmt=".0f",
    heatmap_xlabel="Investment / emission avoided (euro/tCO2)",
    swarm_xlim=(0, 2000),
)

run_interaction_suite(
    metric="Investment / energy savings (euro/kWh)",
    swarm_file=os.path.join(folder_out, "assessment_swarmplot_cost_efficiency_kwh.pdf"),
    swarm_xlabel="Investment / energy savings (euro/kWh)",
    heatmap_file=os.path.join(folder_out, "interactions_cost_efficiency_kwh.pdf"),
    heatmap_fmt=".2f",
    heatmap_xlabel="Investment / energy savings (euro/kWh)",
    swarm_xlim=(0, 1),
)


## Sensitivity Analysis (Sobol + Shapley)


In [ ]:
outcomes = [
    "Cumulated emission saving (MtCO2)",
    "Cumulated energy saving (TWh)",
    "NPV annual (Billion euro/year)",
]
NAME_COLUMNS = None


def prepare_sensitivity_frame():
    sensitivity_df = data.copy()
    active_features = list_features.copy()

    if assessment == "policies":
        rename_columns = {
            "subsidies": "Subsidies",
            "restriction_gas": "Ban gas",
            "cee": "WCO",
            "zero_interest_loan": "Zero interest loan",
            "obligation": "Rental ban",
            "carbon_tax": "Carbon tax",
        }
        active_features = [rename_columns[column] for column in list_features]
        sensitivity_df = sensitivity_df.rename(columns=rename_columns)

    return sensitivity_df, active_features


def run_sensitivity_analysis(analysis_fn, columns, file_prefix):
    for outcome in outcomes:
        sensitivity_result = analysis_fn(df, list(active_features), outcome)

        if NAME_COLUMNS is not None:
            sensitivity_result = sensitivity_result.rename(index=NAME_COLUMNS)
        else:
            sensitivity_result.index = sensitivity_result.index.map(
                lambda value: value.replace("_", " ").capitalize()
            )

        outcome_name = outcome.replace(" ", "_").split("(")[0]
        if assessment == "market_failures":
            title = f"Most influencial market-failures on {outcome.split('(')[0]}"
        elif assessment in ["optimal_pp", "policies"]:
            title = f"Most influencial policies on {outcome.split('(')[0]}"
        else:
            title = outcome

        save_path = os.path.join(folder_out, f"{file_prefix}_{outcome_name}.pdf")

        # Export data to CSV
        csv_path = os.path.splitext(save_path)[0] + ".csv"
        sensitivity_result.to_csv(csv_path)

        horizontal_stack_bar_plot(
            sensitivity_result,
            columns=columns,
            title=title,
            order=columns[-1],
            save_path=save_path,
        )


df, active_features = prepare_sensitivity_frame()


In [ ]:
run_sensitivity_analysis(
    analysis_fn=manual_sobol_analysis,
    columns=["First order", "Total order"],
    file_prefix="sobol",
)

run_sensitivity_analysis(
    analysis_fn=manual_shapley_analysis,
    columns=["Shapley share"],
    file_prefix="shapley",
)

# Shapley waterfall charts
for outcome in outcomes:
    result = manual_shapley_analysis(df, list(active_features), outcome)

    if NAME_COLUMNS is not None:
        result = result.rename(index=NAME_COLUMNS)
    else:
        result.index = result.index.map(
            lambda value: value.replace("_", " ").capitalize()
        )

    outcome_name = outcome.replace(" ", "_").split("(")[0]
    unit = outcome.split("(")[-1].rstrip(")")
    save_path = os.path.join(folder_out, f"shapley_waterfall_{outcome_name}.pdf")

    if assessment == "market_failures":
        vertical_waterfall_chart(
            result["Shapley value"],
            save_path=save_path,
            ylabel=unit,
        )
    else:
        horizontal_waterfall_chart(
            result["Shapley value"],
            save_path=save_path,
            ylabel=unit,
        )

    # Export CSV
    csv_path = os.path.splitext(save_path)[0] + ".csv"
    export = result[["Shapley value", "Shapley share"]].sort_values("Shapley value")
    export.loc["Total", "Shapley value"] = export["Shapley value"].sum()
    export.loc["Total", "Shapley share"] = 1.0
    export.to_csv(csv_path)

## Specific Scenario Comparison

Build focused comparisons by keeping only non-`Other` grouped scenarios.


In [ ]:
if assessment == "market_failures":
    reference = "All friction"
elif assessment in ["policies", "optimal_pp"]:
    reference = "No policy"

scenarios = data.index[data["Group"] != "Other"]
dict_output_subset = {name: output for name, output in dict_output.items() if name in scenarios}

grouped_outputs = {data.loc[name, "Group"]: output for name, output in dict_output_subset.items()}
config_policies = get_json("project/input/policies/cba_inputs.json")
order_scenarios = None
policy_figure_ext = ".pdf" if assessment == "market_failures" else ".png"
policy_horizontal_fontxtick = 10 if assessment == "market_failures" else 12
policy_horizontal_figure_kwargs = {}
if assessment == "market_failures":
    policy_horizontal_figure_kwargs = {
        "legend_exclude": ["Opportunity cost"],
        "annotation_fontsize": 9,
        "legend_fontsize": 11,
    }

if assessment == "policies":
    exclude = [
        "Package 2018",
        "Package 2021",
        "Baseline Package",
        "Baseline Package + Ban",
        "P2018",
        "P2021",
        "P2024",
        "P2024 + Ban",
        "No friction",
        "SCC Benchmark",
    ]
    grouped_outputs = {key: value for key, value in grouped_outputs.items() if key not in exclude}
    orders = [
        "Carbon tax",
        "Subsidies 2018",
        "Subsidies 2021",
        "Subsidies 2024",
        "WCO 2018",
        "WCO 2021",
        "WCO 2024",
        "Zero interest loan",
        "Rental ban",
        "Ban gas",
    ]
    order_scenarios = [key for key in orders if key in grouped_outputs]
    reference = "No policy"

if assessment == "market_failures":
    grouped_outputs = {key: value for key, value in grouped_outputs.items() if key not in ["No friction"]}

if assessment == "optimal_pp":
    remove = ["No friction", "Second best subsidies", "Baseline Package"]
    grouped_outputs = {key: value for key, value in grouped_outputs.items() if key not in remove}

    grouped_outputs_all = {data.loc[name, "Group"]: output for name, output in dict_output_subset.items()}
    package_outputs = {
        key: value
        for key, value in grouped_outputs_all.items()
        if key in remove + [reference]
    }
    _ = indicator_policies(
        package_outputs,
        folder,
        config_policies,
        reference=reference,
        figure=True,
        order_scenarios=remove,
        figure_ext=policy_figure_ext,
        horizontal_fontxtick=policy_horizontal_fontxtick,
        horizontal_figure_kwargs=policy_horizontal_figure_kwargs,
    )

_ = indicator_policies(
    grouped_outputs,
    folder,
    config_policies,
    reference=reference,
    figure=True,
    order_scenarios=order_scenarios,
    figure_ext=policy_figure_ext,
    horizontal_fontxtick=policy_horizontal_fontxtick,
    horizontal_figure_kwargs=policy_horizontal_figure_kwargs,
)

if assessment == "policies":
    latex_table_path = export_policy_latex_table(
        grouped_outputs,
        folder,
        config_policies,
        reference=reference,
        order_scenarios=[
            "Carbon tax",
            "Subsidies 2024",
            "WCO 2024",
            "Zero interest loan",
            "Rental ban",
            "Ban gas",
        ],
        save_name="summary_assessment_table.txt",
    )
    print(f"LaTeX table exported to: {latex_table_path}")


## Future Improvements

- Add explicit reference scenario charts (`No policy` baseline panel).
- Add dedicated panels for 2018, 2021, and 2024 policy packages.
- Add first-best / second-best / optimal scenario storylines.
- Merge credit-constraint heater and insulation variants when needed.
